# 02 — Geometry Encoding

Demonstrate PointNet++ and DGCNN encoders on 3D point clouds.

**Topics:**
- Model architecture overview
- Forward pass and embedding extraction
- Embedding space visualization (t-SNE)
- Classification performance

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from geofusion.models.pointnet2 import PointNet2Encoder, PointNet2Classifier
from geofusion.models.gnn_encoder import DGCNNEncoder

## PointNet++ Architecture

In [ ]:
encoder = PointNet2Encoder(embed_dim=256, use_normals=False)
print(f'PointNet++ parameters: {sum(p.numel() for p in encoder.parameters()):,}')

# Forward pass with random data
x = torch.randn(4, 1024, 3)
with torch.no_grad():
    embedding = encoder(x)
print(f'Input shape:  {x.shape}')
print(f'Output shape: {embedding.shape}')

## DGCNN Architecture

In [ ]:
dgcnn = DGCNNEncoder(in_channels=3, embed_dim=256, k=20)
print(f'DGCNN parameters: {sum(p.numel() for p in dgcnn.parameters()):,}')

with torch.no_grad():
    dgcnn_emb = dgcnn(x)
print(f'DGCNN output shape: {dgcnn_emb.shape}')

## Embedding Visualization (Synthetic)

In [ ]:
# Generate synthetic embeddings for visualization
n_samples = 200
n_classes = 5
class_names = ['Airplane', 'Chair', 'Car', 'Table', 'Lamp']

encoder = PointNet2Encoder(embed_dim=128, use_normals=False)
encoder.eval()

embeddings = []
labels = []
with torch.no_grad():
    for c in range(n_classes):
        # Different random distributions per "class" for illustration
        pts = torch.randn(n_samples // n_classes, 256, 3) * (1 + c * 0.3)
        emb = encoder(pts)
        embeddings.append(emb.numpy())
        labels.extend([c] * (n_samples // n_classes))

embeddings = np.concatenate(embeddings)
labels = np.array(labels)

# t-SNE
tsne = TSNE(n_components=2, perplexity=20, random_state=42)
emb_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(10, 8))
for c in range(n_classes):
    mask = labels == c
    plt.scatter(emb_2d[mask, 0], emb_2d[mask, 1], label=class_names[c], alpha=0.7, s=30)
plt.legend(fontsize=12)
plt.title('PointNet++ Embedding Space (t-SNE)', fontsize=14)
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

## Classifier Forward Pass

In [ ]:
classifier = PointNet2Classifier(num_classes=40, embed_dim=256, use_normals=False)
x = torch.randn(2, 1024, 3)

with torch.no_grad():
    logits, emb = classifier(x)

probs = torch.softmax(logits, dim=1)
print(f'Top-5 predictions (sample 0):')
top5 = probs[0].topk(5)
for score, idx in zip(top5.values, top5.indices):
    print(f'  Class {idx.item():2d}: {score.item():.4f}')